In [1]:
import polars as pl
import numpy as np
import math
import plotly.express as px

In [2]:
df = pl.read_excel("./data/task2.xlsx", sheet_name="В-18")

df.head()

Х1,Х2,Х3,Х4,Х5,Х6,Х7,Х8,Х9,Y
i64,f64,f64,f64,f64,i64,f64,f64,f64,f64
63,82.787941,479.22016,234.368287,314.954529,41,173.541475,228.767113,284.431898,7098.74738
31,108.811008,754.38713,158.621085,261.953308,178,236.837209,159.587744,300.041229,13014.307845
36,90.300346,478.592271,90.131814,236.620393,38,226.652648,231.73432,295.117184,8748.641969
36,91.904692,719.830328,217.716565,412.566963,158,228.631988,172.016608,296.762484,11394.990417
72,90.251698,657.428365,193.847045,332.99756,128,157.035283,186.7129,281.550313,10154.04808


In [3]:
df.describe()

statistic,Х1,Х2,Х3,Х4,Х5,Х6,Х7,Х8,Х9,Y
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",400.0,400.0,400.0,400.0,400.0,400.0,400.0,400.0,400.0,400.0
"""null_count""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",50.97,99.801038,657.024349,152.00945,298.290133,128.9,197.899886,185.603806,289.439978,10920.12504
"""std""",27.444162,15.808473,149.796361,41.42786,82.596431,74.844433,54.779154,37.569878,13.765893,2647.88559
"""min""",1.0,49.570253,402.489561,47.649812,92.325761,1.0,100.732909,124.557276,262.450023,5131.805871
"""25%""",29.0,89.10881,521.221214,123.092325,242.472639,61.0,153.076708,151.778647,278.188869,8649.632451
"""50%""",51.0,99.291521,660.458752,151.722231,300.252268,133.0,198.759303,183.940731,288.980345,11076.44901
"""75%""",74.0,110.081673,789.491008,181.551927,352.446694,195.0,240.895284,219.660763,300.406114,13014.307845
"""max""",99.0,148.304191,897.209732,295.972554,522.329574,249.0,298.124643,250.877451,316.563645,16017.916121


In [4]:
def get_col_stats(col: pl.Series) -> dict:
    n = col.len()
    min = col.min()
    max = col.max()
    r = max - min
    k = 1 + math.floor(3.222 * np.log10(n)) # правило Стёрджеса с округлением вниз
    h = r / k

    result = {
        "n": n,
        "min": min,
        "max": max,
        "r": r,
        "k": k,
        "h": h
    }

    return result

In [5]:
def get_bins(col_stats: dict) -> np.ndarray:
    
    min_val = col_stats["min"]
    h = col_stats["h"]
    k = col_stats["k"]
    max_val = col_stats["max"]

    bins = min_val + np.arange(k) * h
    bins = np.append(bins, max_val)
    
    return bins

In [6]:
def get_interval_table(col: pl.Series) -> pl.DataFrame:
    stats = get_col_stats(col)
    bins = get_bins(stats)

    groups = np.digitize(col.to_numpy(), bins[1:-1], right=False)

    interval_labels = []
    for i in range(stats["k"]):
        left = bins[i]
        right = bins[i + 1]

        if i < stats["k"] - 1:
            label = f"[{left:.4g}, {right:.4g})"
        else:
            label = f"[{left:.4g}, {right:.4g}]"
        interval_labels.append(label)

    df = pl.DataFrame({
        "value": col,
        "group": groups
    })

    group_stats = (
        df.group_by("group")
        .agg(
            pl.len().alias("freq"),
            pl.col("value").mean().alias("mean")
        )
        .with_columns(
            (pl.col("freq") / len(df)).alias("w")
        )
        .sort("group")
    )

    group_stats = group_stats.with_columns(
        pl.col("freq").cum_sum().alias("cum_freq"),
        pl.col("w").cum_sum().alias("cum_w")
    )

    result = pl.DataFrame({
        "group": list(range(stats["k"])),
        "interval": interval_labels,
    })

    result = result.join(group_stats, on="group", how="left")

    result = result.select(["interval", "freq", "w", "cum_freq", "cum_w", "mean"])
   
    return result

In [18]:
def plot_freq_bar(interval_table: pl.DataFrame, title: str) -> None:
    fig = px.bar(
        interval_table,
        x="interval", 
        y="freq", 
        title=f"Гистограмма частот для {title}", 
        labels={
            "interval": "Интервалы", 
            "freq": "Частота"
        },
        width=800,
        height=400,
        text_auto=True
    )
    
    fig.show()

In [27]:
def plot_polygon(interval_table: pl.DataFrame, title: str) -> None:
    fig = px.line(
        interval_table,
        x="interval", 
        y="cum_w", 
        title=f"Полигон частот для {title}", 
        labels={
            "interval": "Интервалы", 
            "freq": "Частота"
        },
        width=800,
        height=400,
        markers=True
    )
    
    fig.show()

# Задание 1

## X1

In [7]:
x1 = df.get_column("Х1")

In [8]:
x1_stats = get_col_stats(x1)
x1_stats

{'n': 400, 'min': 1, 'max': 99, 'r': 98, 'k': 9, 'h': 10.88888888888889}

In [9]:
x1_bins = get_bins(x1_stats)
x1_bins

array([ 1.        , 11.88888889, 22.77777778, 33.66666667, 44.55555556,
       55.44444444, 66.33333333, 77.22222222, 88.11111111, 99.        ])

In [10]:
x1_intervals = get_interval_table(x1)
x1_intervals

interval,freq,w,cum_freq,cum_w,mean
str,u32,f64,u32,f64,f64
"""[1, 11.89)""",39,0.0975,39,0.0975,6.358974
"""[11.89, 22.78)""",34,0.085,73,0.1825,17.117647
"""[22.78, 33.67)""",47,0.1175,120,0.3,27.93617
"""[33.67, 44.56)""",50,0.125,170,0.425,38.18
"""[44.56, 55.44)""",47,0.1175,217,0.5425,49.404255
"""[55.44, 66.33)""",55,0.1375,272,0.68,61.854545
"""[66.33, 77.22)""",36,0.09,308,0.77,71.138889
"""[77.22, 88.11)""",52,0.13,360,0.9,82.788462
"""[88.11, 99]""",40,0.1,400,1.0,93.65


In [19]:
plot_freq_bar(x1_intervals, "Х1")

In [ ]:
# Доделать
plot_polygon(x1_intervals, "Х1")